# Flower Recognition — GreenTech Solutions Ltd.

**Progetto:** Classificazione binaria Daisy vs Dandelion  
**Obiettivo:** Massimizzare F1-score macro sul test set  
**Stack:** PyTorch Lightning + timm + Albumentations

---

## Contesto

GreenTech Solutions Ltd. richiede un sistema di riconoscimento automatico dei fiori per ottimizzare le procedure agricole, monitorare la salute vegetale e supportare decisioni basate sui dati. Il dataset contiene due classi: **Daisy** (Margherita) e **Dandelion** (Tarassaco), con un leggero sbilanciamento a favore del Dandelion (~41% in più).

La **metrica target** è l'F1-score macro: bilancia precision e recall su entrambe le classi e compensa lo sbilanciamento meglio dell'accuracy.

**Flusso metodologico (rigido):**

```
TRAIN → VALIDATION → scelta backbone → threshold tuning → MODELLO DEFINITIVO → TEST → (TTA opzionale)
```

Il test set NON è usato per selezionare il modello. La TTA è solo analisi finale opzionale.

## 1. Installazione dipendenze

- **torch / torchvision**: Backend PyTorch per deep learning e gestione immagini
- **pytorch-lightning**: Framework per training strutturato; gestisce callbacks, multi-device, logging
- **timm**: Libreria di modelli pre-trained (EfficientNet, ResNet, ConvNeXt, ecc.) con preprocessing standardizzato
- **albumentations**: Data augmentation avanzata per migliorare la generalizzazione
- **grad-cam**: Visualizzazione interpretabilità (GradCAM)

In [ ]:
!pip install -q torch torchvision pytorch-lightning timm albumentations grad-cam matplotlib seaborn

## 2. Imports

In [ ]:
import os
import warnings

# Soppressione warning noti — PRIMA di importare pytorch_lightning
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"pytorch_lightning\.utilities")
warnings.filterwarnings("ignore", module="pytorch_lightning.trainer.connectors.data_connector")
warnings.filterwarnings("ignore", message=".*pin_memory.*MPS.*", category=UserWarning)
warnings.filterwarnings("ignore", module="pytorch_lightning.loops.fit_loop")

import torch
import timm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision.datasets import ImageFolder
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import albumentations as A
from albumentations.pytorch import ToTensorV2
from dataclasses import dataclass, field, asdict
from typing import List, Optional

# Riproducibilità garantita su tutti i run
pl.seed_everything(42, workers=True)

## 3. Config e hyperparameters

Configurazione centralizzata con `dataclass` (serializzabile, type-safe, compatibile con `save_hyperparameters` di Lightning).

In [ ]:
@dataclass
class Config:
    # Paths
    DATA_URL: str = "https://proai-datasets.s3.eu-west-3.amazonaws.com/progetto-finale-flowes.tar.gz"
    DATA_DIR: str = "./data"
    CHECKPOINT_DIR: str = "./checkpoints"

    # Training
    BATCH_SIZE: int = 32
    MAX_EPOCHS: int = 25
    NUM_WORKERS: int = 0  # 0 per compatibilità MPS (Apple Silicon); warning soppresso in imports
    IMG_SIZE: int = 224

    # Model — Ablation study
    BACKBONES: List[str] = field(default_factory=lambda: ["resnet50", "efficientnet_b0", "efficientnet_b2", "convnext_tiny"])
    # BACKBONES: List[str] = field(default_factory=lambda: ["efficientnet_b0"])  # Per test rapido: ["efficientnet_b0"]
    BACKBONE: Optional[str] = None  # Impostato dinamicamente nel loop
    DROPOUT: float = 0.3

    # Learning rates
    LR_HEAD: float = 1e-3          # Fase 1: solo head (backbone frozen)
    LR_BACKBONE: float = 1e-5      # Fase 2: backbone (discriminative LR)
    LR_HEAD_FINETUNE: float = 1e-4 # Fase 2: head
    WEIGHT_DECAY: float = 1e-4

    # Fasi di training
    FROZEN_EPOCHS: int = 5         # Epoche con backbone frozen

    # Checkpoint
    SAVE_EVERY_N_EPOCHS: int = 2
    SAVE_TOP_K_PERIODIC: int = 3
    RESUME_CKPT: Optional[str] = None
    RESUME_BACKBONE: Optional[str] = None

# Istanza globale
cfg_global = Config()

## 4. Colab setup — GPU, download dataset, mount Drive

In [ ]:
# Verifica acceleratore
if torch.cuda.is_available():
    print(f"GPU NVIDIA disponibile: {torch.cuda.get_device_name(0)}")
    accelerator = "gpu"
    # ATTENZIONE:MPS (Apple Silicon) non supporta pienamente float16 — si usa solo con GPU NVIDIA
    precision = "16-mixed"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    print("Apple Silicon (MPS) disponibile — precision forzata a 32 per stabilità")
    accelerator = "mps"
    precision = "32"  # ATTENZIONE: 16-mixed non è garantito su MPS quindi si usa 32
else:
    print("Nessun acceleratore hardware — training su CPU (più lento)")
    accelerator = "cpu"
    precision = "32"

print(f"Acceleratore: {accelerator} | Precision: {precision}")

In [ ]:
# Download e estrazione dataset
import urllib.request
import tarfile
import shutil

os.makedirs(cfg_global.DATA_DIR, exist_ok=True)
archive_path = os.path.join(cfg_global.DATA_DIR, "progetto-finale-flowes.tar.gz")

if not os.path.exists(archive_path):
    print("Download dataset in corso...")
    urllib.request.urlretrieve(cfg_global.DATA_URL, archive_path)
    print("Download completato.")

if not os.path.exists(os.path.join(cfg_global.DATA_DIR, "train")):
    print("Estrazione archivio...")
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(cfg_global.DATA_DIR)
    items = os.listdir(cfg_global.DATA_DIR)
    if "train" not in items:
        subdirs = [x for x in items if os.path.isdir(os.path.join(cfg_global.DATA_DIR, x))]
        sub = os.path.join(cfg_global.DATA_DIR, subdirs[0])
        for name in os.listdir(sub):
            shutil.move(os.path.join(sub, name), os.path.join(cfg_global.DATA_DIR, name))
        os.rmdir(sub)
    print("Estrazione completata.")

train_path = os.path.join(cfg_global.DATA_DIR, "train")
val_path   = os.path.join(cfg_global.DATA_DIR, "val")
test_path  = os.path.join(cfg_global.DATA_DIR, "test")
print(f"Train: {train_path} — exists: {os.path.exists(train_path)}")
print(f"Val:   {val_path}   — exists: {os.path.exists(val_path)}")
print(f"Test:  {test_path}  — exists: {os.path.exists(test_path)}")

In [ ]:
# Rimozione file spazzatura dopo il donwload e l'estrazione dell'archivio
import os

removed = 0
search_root = os.getcwd()

for dirpath, dirnames, filenames in os.walk(search_root):
    for filename in filenames:
        if filename.startswith("._") or filename == ".DS_Store":
            full_path = os.path.join(dirpath, filename)
            try:
                os.remove(full_path)
                removed += 1
            except Exception as e:
                print(f"Errore: {full_path}: {e}")

print(f"File spazzatura rimossi: {removed}")

In [ ]:
# FIX: sembra che il dataset usi "valid" non "val" — facciamo un rename per compatibilità col codice
import shutil

valid_src = os.path.join(cfg_global.DATA_DIR, "valid")
val_dst   = os.path.join(cfg_global.DATA_DIR, "val")

if os.path.exists(valid_src) and not os.path.exists(val_dst):
    shutil.move(valid_src, val_dst)
    print(f"Rinominato: {valid_src} → {val_dst}")
else:
    print(f"val path: {val_dst} — exists: {os.path.exists(val_dst)}")

In [ ]:
# Opzionale: mount Google Drive per checkpoint persistenti
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cfg_global.CHECKPOINT_DIR = "/content/drive/MyDrive/flowers_checkpoints"
    print(f"Checkpoint salvati su: {cfg_global.CHECKPOINT_DIR}")

os.makedirs(cfg_global.CHECKPOINT_DIR, exist_ok=True)

## 5. EDA — Esplorazione dati

Analisi esplorativa con:
- Conteggi per classe e split
- Visualizzazione dello sbilanciamento
- Sample grid di immagini per classe
- Distribuzione delle dimensioni delle immagini

In [ ]:
# Conteggi per split e classe (SOLO train e val — il test non va usato in EDA per evitare peek)
train_ds_eda = ImageFolder(train_path)
val_ds_eda   = ImageFolder(val_path)

classes = train_ds_eda.classes
print(f"Classi (ordine alfabetico): {classes}")
print()

splits = {"Train": train_ds_eda, "Val": val_ds_eda}
counts = {}
for split_name, ds in splits.items():
    c = [len([x for x in ds.samples if x[1] == i]) for i in range(len(classes))]
    counts[split_name] = c
    print(f"{split_name:6}: {classes[0]}={c[0]:4d}  {classes[1]}={c[1]:4d}  Total={sum(c)}")
print("(Test set escluso dall'EDA — usato solo nella valutazione finale)")

# Calcolo dell'Imbalance Ratio (IR) sul train set
# Più in generale un IR > 10 solitamente indica un dataset sbilanciato che richiede tecniche
# di resampling (oversampling/undersampling) o class_weighting.
# Definizione: IR = count(majority_class) / count(minority_class)

ratio = counts["Train"][1] / counts["Train"][0]
print(f"\nImbalance ratio (Dandelion/Daisy): {ratio:.2f}x")

In [ ]:
# Visualizzazione sbilanciamento (solo Train e Val)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colors = ["#4CAF50", "#FF9800"]

for ax, (split_name, c) in zip(axes, counts.items()):
    bars = ax.bar(classes, c, color=colors, edgecolor="white", linewidth=1.5)
    ax.set_title(f"{split_name} set", fontsize=13, fontweight="bold")
    ax.set_ylabel("Numero immagini")
    for bar, val in zip(bars, c):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(val), ha="center", va="bottom", fontweight="bold")
    ax.set_ylim(0, max(c) * 1.2)

plt.suptitle("Distribuzione classi per split", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Sample grid per classe — con filtro file macOS metadata (._*)
from PIL import Image, UnidentifiedImageError

def is_valid_image(path):
    """Filtra file ._* (macOS metadata) e file non leggibili da PIL."""
    if os.path.basename(path).startswith("._"):
        return False
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except (UnidentifiedImageError, Exception):
        return False

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for class_idx, class_name in enumerate(classes):
    # FIX: filtra i file invalidi prima di selezionare i sample
    all_paths = [s[0] for s in train_ds_eda.samples if s[1] == class_idx]
    valid_paths = [p for p in all_paths if is_valid_image(p)][:5]
    for col, path in enumerate(valid_paths):
        img = Image.open(path).resize((150, 150))
        axes[class_idx][col].imshow(img)
        axes[class_idx][col].axis("off")
        if col == 0:
            axes[class_idx][col].set_ylabel(class_name, fontsize=12, fontweight="bold", labelpad=10)
            axes[class_idx][col].yaxis.set_label_position("left")
            axes[class_idx][col].set_yticks([])

plt.suptitle("Sample immagini per classe", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Distribuzione dimensioni immagini — con filtro file macOS metadata (._*)
import random

widths, heights, img_classes = [], [], []
for class_idx, class_name in enumerate(classes):
    all_paths = [s[0] for s in train_ds_eda.samples if s[1] == class_idx]
    # FIX: filtra ._* prima del random.sample
    valid_paths = [p for p in all_paths if not os.path.basename(p).startswith("._")]
    sample_paths = random.sample(valid_paths, min(50, len(valid_paths)))
    for path in sample_paths:
        with Image.open(path) as img:
            w, h = img.size
        widths.append(w)
        heights.append(h)
        img_classes.append(class_name)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, dim, label in zip(axes, [widths, heights], ["Width", "Height"]):
    for class_name, color in zip(classes, colors):
        vals = [v for v, c in zip(dim, img_classes) if c == class_name]
        ax.hist(vals, bins=20, alpha=0.6, color=color, label=class_name)
    ax.set_title(f"Distribuzione {label}", fontweight="bold")
    ax.set_xlabel("Pixel")
    ax.legend()

plt.suptitle("Distribuzione dimensioni immagini (campione)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. FlowerDataModule

### Pipeline augmentation

**Train:** `RandomResizedCrop`, flip, rotazioni, `ColorJitter`, `GaussianBlur` per aumentare la variabilità.

**Val/Test:** solo `Resize` + `CenterCrop` + normalizzazione. Pipeline conservativa; soggetti decentrati potrebbero essere parzialmente tagliati.

> I parametri `mean` e `std` vengono estratti automaticamente da `timm.data.resolve_model_data_config()` per ogni backbone.

### Sbilanciamento — strategia singola

Si usa **solo `WeightedRandomSampler`** (dopo alcune verifiche preliminari ho eliminato `pos_weight` nella loss che compensava la classe sbagliata). Strategia più pulita e senza rischio di doppia correzione conflittuale.

In [ ]:
def get_train_transform(img_size, mean, std):
    return A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.8, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),  # ATTENZIONE -> Sperimentale: meno realistico di H-flip per fiori
        A.Rotate(limit=30, p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
        A.GaussianBlur(blur_limit=(3, 5), p=0.3),
        A.Normalize(mean=mean, std=std),
        ToTensorV2()
    ])

def get_val_transform(img_size, mean, std):
    # Pipeline val/test senza CenterCrop per evitare il taglio di soggetti decentrati
    return A.Compose([
        A.Resize(height=img_size, width=img_size),
        A.Normalize(mean=mean, std=std),
        ToTensorV2()
    ])


class FlowerDataset(torch.utils.data.Dataset):
    def __init__(self, root, transform=None):
        base = ImageFolder(root)
        # FIX: anche qui rimuoviamo file ._* (macOS metadata) dalla lista samples
        self.dataset = base
        self.dataset.samples = [
            (p, l) for p, l in base.samples
            if not os.path.basename(p).startswith("._")
        ]
        self.dataset.imgs = self.dataset.samples  # ImageFolder usa anche .imgs
        self.transform = transform

    def __len__(self):
        return len(self.dataset.samples)

    def __getitem__(self, idx):
        path, label = self.dataset.samples[idx]
        img = Image.open(path).convert("RGB")  # .convert("RGB") gestisce anche PNG con alpha
        img = np.array(img)
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, label


class FlowerDataModule(pl.LightningDataModule):

    def __init__(self, config: Config, mean: list, std: list):
        super().__init__()
        self.config = config
        self.mean = mean
        self.std = std

    def setup(self, stage=None):
        train_path = os.path.join(self.config.DATA_DIR, "train")
        val_path   = os.path.join(self.config.DATA_DIR, "val")
        test_path  = os.path.join(self.config.DATA_DIR, "test")

        self.train_ds = FlowerDataset(train_path, get_train_transform(self.config.IMG_SIZE, self.mean, self.std))
        self.val_ds   = FlowerDataset(val_path,   get_val_transform(self.config.IMG_SIZE, self.mean, self.std))
        self.test_ds  = FlowerDataset(test_path,  get_val_transform(self.config.IMG_SIZE, self.mean, self.std))

        # FIX: come gia anticipato usiamo solo WeightedRandomSampler (rimosso pos_weight che upweightava la classe errata)
        class_counts = np.bincount([y for _, y in self.train_ds.dataset.samples])
        # Peso per campione = inverso della frequenza della sua classe
        sample_weights = np.array([
            1.0 / class_counts[y] for _, y in self.train_ds.dataset.samples
        ])
        self.sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )
        # class_weights esposto per eventuali usi downstream (es. metriche pesate)
        self.class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32)

    def train_dataloader(self):
        pin_mem = torch.cuda.is_available()  # False su MPS/CPU — evita warning pin_memory
        return DataLoader(
            self.train_ds, batch_size=self.config.BATCH_SIZE,
            sampler=self.sampler, num_workers=self.config.NUM_WORKERS,
            pin_memory=pin_mem, persistent_workers=(self.config.NUM_WORKERS > 0)
        )

    def val_dataloader(self):
        pin_mem = torch.cuda.is_available()
        return DataLoader(
            self.val_ds, batch_size=self.config.BATCH_SIZE, shuffle=False,
            num_workers=self.config.NUM_WORKERS,
            pin_memory=pin_mem, persistent_workers=(self.config.NUM_WORKERS > 0)
        )

    def test_dataloader(self):
        pin_mem = torch.cuda.is_available()
        return DataLoader(
            self.test_ds, batch_size=self.config.BATCH_SIZE, shuffle=False,
            num_workers=self.config.NUM_WORKERS,
            pin_memory=pin_mem, persistent_workers=(self.config.NUM_WORKERS > 0)
        )


def get_dm_and_config(backbone_name: str, base_config: Config) -> tuple:
    """Crea DataModule e Config per un dato backbone, usando mean/std corretti da timm."""
    # Usa pretrained=False per non scaricare pesi solo per la config
    tmp_model = timm.create_model(backbone_name, pretrained=False)
    model_cfg = timm.data.resolve_model_data_config(tmp_model)
    del tmp_model
    mean = list(model_cfg["mean"])
    std  = list(model_cfg["std"])

    # FIX: Config è una dataclass — copia con asdict e ricrea, compatibile con Lightning
    cfg_dict = asdict(base_config)
    cfg_dict["BACKBONE"] = backbone_name
    cfg = Config(**cfg_dict)

    dm = FlowerDataModule(cfg, mean, std)
    dm.setup()
    return dm, cfg

## 7. FlowerClassifier (LightningModule)

1. **`test_step` + `on_test_epoch_end`** implementati correttamente  
2. **Scheduler compatibile con unfreeze** — `LambdaLR` con due fasi (frozen head, poi fine-tuning)  
3. **`f1_score` rimosso dal `training_step`** — accumulazione come in validation, calcolo in `on_train_epoch_end`  
4. **`reshape(-1)` + `feat.contiguous()`** — evita RuntimeError poichè "view size is not compatible" su MPS/mixed precision  
5. **`save_hyperparameters` con config serializzabile** — la dataclass è compatibile con il checkpoint Lightning

In [ ]:
class FlowerClassifier(pl.LightningModule):

    def __init__(self, config: Config):
        super().__init__()
        self.config = config
        self.save_hyperparameters({"config": asdict(config)})

        self.backbone = timm.create_model(config.BACKBONE, pretrained=True, num_classes=0)
        feat_dim = self.backbone.num_features
        self.classifier = torch.nn.Sequential(
            torch.nn.Dropout(config.DROPOUT),
            torch.nn.Linear(feat_dim, 1)
        )

        self._frozen = True
        # Accumulatori per metriche epoch-level (val, test, train)
        self.train_preds, self.train_labels = [], []
        self.val_preds,   self.val_labels   = [], []
        self.test_preds,  self.test_labels  = [], []

    # ------------------------------------------------------------------ forward
    def forward(self, x):
        feat = self.backbone(x).contiguous()  # Garantisce contiguità (fix MPS/mixed precision)
        # reshape(-1) invece di squeeze — evita RuntimeError "view size is not compatible"
        return self.classifier(feat).reshape(-1)

    # ----------------------------------------------------------- freeze/unfreeze
    def _freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False
        self._frozen = True

    def _unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True
        self._frozen = False

    # ------------------------------------------------------- training step/epoch
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        # BCEWithLogitsLoss senza pos_weight (sbilanciamento gestito da WeightedRandomSampler)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y.float())
        preds = (torch.sigmoid(logits) > 0.5).long()
        self.train_preds.append(preds.detach().cpu())
        self.train_labels.append(y.detach().cpu())
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def on_train_epoch_end(self):
        if self.train_preds:
            preds  = torch.cat(self.train_preds).numpy()
            labels = torch.cat(self.train_labels).numpy()
            f1 = f1_score(labels, preds, average="macro", zero_division=0)
            self.log("train_f1_macro", f1, prog_bar=True)
            self.train_preds.clear()
            self.train_labels.clear()

    # ----------------------------------------------------- validation step/epoch
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y.float())
        preds = (torch.sigmoid(logits) > 0.5).long()
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.val_preds.append(preds.detach().cpu())
        self.val_labels.append(y.detach().cpu())
        return loss

    def on_validation_epoch_end(self):
        if self.val_preds:
            preds  = torch.cat(self.val_preds).numpy()
            labels = torch.cat(self.val_labels).numpy()
            f1 = f1_score(labels, preds, average="macro", zero_division=0)
            self.log("val_f1_macro", f1, prog_bar=True)
            self.val_preds.clear()
            self.val_labels.clear()

    # ----------------------------------------------------------- test step/epoch
    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y.float())
        preds = (torch.sigmoid(logits) > 0.5).long()
        self.log("test_loss", loss, on_step=False, on_epoch=True)
        self.test_preds.append(preds.detach().cpu())
        self.test_labels.append(y.detach().cpu())
        return loss

    def on_test_epoch_end(self):
        if self.test_preds:
            preds  = torch.cat(self.test_preds).numpy()
            labels = torch.cat(self.test_labels).numpy()
            f1 = f1_score(labels, preds, average="macro", zero_division=0)
            self.log("test_f1_macro", f1)
            # Salva per accesso esterno (valutazione dettagliata)
            self._last_test_preds  = preds
            self._last_test_labels = labels
            self.test_preds.clear()
            self.test_labels.clear()

    # ---------------------------------------------------- optimizer e scheduler
    def configure_optimizers(self):
        """
        LambdaLR con due fasi: frozen head (epoch 0→FROZEN_EPOCHS-1),
        poi fine-tuning con LR discriminativi (backbone + head).
                                             CosineAnnealingLR su FROZEN_EPOCHS step.
        Fase 2: backbone + head con LR discriminativi (coseno decrescente).
        Backbone pre-incluso con lr=0 in fase 1, poi scalato da LambdaLR.
        """
        remaining = self.config.MAX_EPOCHS - self.config.FROZEN_EPOCHS

        opt = torch.optim.AdamW([
            # Backbone: lr=0 in fase 1, verrà poi scalato dallo scheduler in fase 2
            {"params": self.backbone.parameters(),   "lr": 0.0},
            {"params": self.classifier.parameters(), "lr": self.config.LR_HEAD},
        ], weight_decay=self.config.WEIGHT_DECAY)


        # Fase 1: solo il head sale (backbone rimane a lr≈0 per FROZEN_EPOCHS)
        # Usiamo LinearLR per il backbone (rimane piatto a 0) e CosineAnnealingLR per il head
        # ATTENZIONE: CosineAnnealingLR non è un scheduler per lr=0 — si usa invece ConstantLR
        sched_frozen_backbone = torch.optim.lr_scheduler.ConstantLR(
            opt, factor=0.0, total_iters=self.config.FROZEN_EPOCHS
        )
        sched_frozen_head = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=self.config.FROZEN_EPOCHS
        )

        # Fase 2: fine-tuning con LR discriminativi
        # Il SequentialLR gestisce la transizione automaticamente
        def lr_lambda_backbone(epoch):
            if epoch < self.config.FROZEN_EPOCHS:
                return 0.0
            frac = (epoch - self.config.FROZEN_EPOCHS) / max(remaining, 1)
            return self.config.LR_BACKBONE * (1 + np.cos(np.pi * frac)) / 2 / self.config.LR_HEAD

        def lr_lambda_head(epoch):
            if epoch < self.config.FROZEN_EPOCHS:
                frac = epoch / max(self.config.FROZEN_EPOCHS, 1)
                return (1 + np.cos(np.pi * frac)) / 2
            frac = (epoch - self.config.FROZEN_EPOCHS) / max(remaining, 1)
            return self.config.LR_HEAD_FINETUNE * (1 + np.cos(np.pi * frac)) / 2 / self.config.LR_HEAD

        scheduler = torch.optim.lr_scheduler.LambdaLR(
            opt, lr_lambda=[lr_lambda_backbone, lr_lambda_head]
        )

        return {"optimizer": opt, "lr_scheduler": {"scheduler": scheduler, "interval": "epoch"}}

    def on_train_epoch_start(self):
        if self.current_epoch == self.config.FROZEN_EPOCHS and self._frozen:
            self._unfreeze_backbone()
            import sys
            sys.stdout.write(
                f"\n[Epoch {self.current_epoch}] Backbone sbloccato — fine-tuning con LR discriminativi\n"
            )
            sys.__stdout__.flush()

## 7b. Funzioni helper per valutazione

Utility riutilizzabili per evitare duplicazione:
- `collect_probs_and_labels`: raccoglie probabilità e label da un loader
- `compute_binary_metrics`: calcola F1 macro e predizioni con una soglia

In [ ]:
def collect_probs_and_labels(model, loader, device):
    """Raccoglie probabilità (sigmoid) e label da un DataLoader. Restituisce (probs, labels)."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            logits = model(x.to(device))
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs.tolist())
            all_labels.extend(y.tolist())
    return np.array(all_probs), np.array(all_labels)


def compute_binary_metrics(labels, probs, threshold=0.5):
    """Calcola predizioni e F1 macro per una soglia. Restituisce (preds, f1_macro)."""
    preds = (probs >= threshold).astype(int)
    f1 = f1_score(labels, preds, average="macro", zero_division=0)
    return preds, f1


def evaluate_loader(model, loader, device, threshold=0.5):
    """Valutazione completa: restituisce probs, labels, preds, f1_macro."""
    probs, labels = collect_probs_and_labels(model, loader, device)
    preds, f1_macro = compute_binary_metrics(labels, probs, threshold)
    return probs, labels, preds, f1_macro

## 8. Ablation Study — Training e confronto su validation

**Flusso metodologico corretto:**
- Ogni backbone viene allenato; il best checkpoint è selezionato tramite `val_f1_macro`
- Il confronto tra modelli avviene **solo sul validation set**
- Il **test set è riservato** alla valutazione finale del solo modello "vincente" (sezione 11)

Per ogni backbone: train → carica best checkpoint → valuta su validation → salva `val_f1_macro`

In [ ]:
results = []
backbones_to_run = cfg_global.BACKBONES

# Modalità resume: esegui solo il backbone specificato
if cfg_global.RESUME_CKPT and cfg_global.RESUME_BACKBONE:
    backbones_to_run = [cfg_global.RESUME_BACKBONE]
    print(f"Modalità resume: eseguo solo {cfg_global.RESUME_BACKBONE}")

for backbone_name in backbones_to_run:
    print(f"\n{'='*60}\n>>> Backbone: {backbone_name}\n{'='*60}")

    dm, cfg = get_dm_and_config(backbone_name, cfg_global)

    ckpt_base = os.path.join(cfg_global.CHECKPOINT_DIR, backbone_name)
    os.makedirs(os.path.join(ckpt_base, "best"),     exist_ok=True)
    os.makedirs(os.path.join(ckpt_base, "periodic"), exist_ok=True)


    callbacks = [
    EarlyStopping(monitor="val_f1_macro", patience=5, mode="max", verbose=True),
    
    # Best checkpoint — invariato
    ModelCheckpoint(
        dirpath=os.path.join(ckpt_base, "best"),
        monitor="val_f1_macro", mode="max", save_top_k=1,
        filename="best-{epoch:02d}-{val_f1_macro:.4f}"
        ),
    
    # Periodic checkpoint — solo l'ultimo, sovrascrive sempre
    ModelCheckpoint(
        dirpath=os.path.join(ckpt_base, "periodic"),
        every_n_epochs=cfg_global.SAVE_EVERY_N_EPOCHS,
        save_top_k=1,           # tieni solo 1 checkpoint
        save_last=True,         # salva sempre l'ultimo
        filename="resume",      # nome fisso → sovrascrive sempre lo stesso file
        ),
    ]


    model = FlowerClassifier(cfg)
    model._freeze_backbone()

    trainer = pl.Trainer(
        max_epochs=cfg_global.MAX_EPOCHS,
        accelerator=accelerator,
        callbacks=callbacks,
        precision=precision,
        gradient_clip_val=1.0,
        log_every_n_steps=10,  # < 40 batch/epoca — evita warning e mostra log
        deterministic=True  # Riproducibilità
    )

    ckpt_path = None
    if cfg_global.RESUME_CKPT and os.path.exists(cfg_global.RESUME_CKPT):
        ckpt_path = cfg_global.RESUME_CKPT
        print(f"Ripresa da checkpoint: {ckpt_path}")

    trainer.fit(model, dm, ckpt_path=ckpt_path)

    # Carica il best checkpoint e valuta SOLO sul validation set (NON sul test)
    best_dir = os.path.join(ckpt_base, "best")
    ckpts = [f for f in os.listdir(best_dir) if f.endswith(".ckpt")]

    if ckpts:
        best_ckpt_path = os.path.join(best_dir, ckpts[0])
        best_model = FlowerClassifier.load_from_checkpoint(best_ckpt_path, config=cfg, weights_only=False)
        device = next(best_model.parameters()).device

        # Valutazione sul VALIDATION set (threshold 0.5)
        _, _, _, val_f1_macro = evaluate_loader(
            best_model, dm.val_dataloader(), device, threshold=0.5
        )

        results.append({
            "backbone":      backbone_name,
            "val_f1_macro":  val_f1_macro,
            "checkpoint":   best_ckpt_path,
            "model":        best_model,
            "dm":           dm,
            "cfg":          cfg
        })
        print(f"F1 macro validation ({backbone_name}): {val_f1_macro:.4f}")

## 9. Riepilogo Ablation — Best backbone on validation

Confronto basato **solo su validation F1**. Il test set non è stato usato per selezionare il modello.

In [ ]:
# FIX: rimossa la riga orfana \" che causava SyntaxError
if results:
    print("\n" + "="*55 + " RIEPILOGO ABLATION STUDY " + "="*55)
    print(f"  {'Backbone':25} | F1 macro validation")
    print("  " + "-"*42)
    for r in sorted(results, key=lambda x: -x["val_f1_macro"]):
        print(f"  {r['backbone']:25} | {r['val_f1_macro']:.4f}")

    best = max(results, key=lambda x: x["val_f1_macro"])
    print(f"\n>>> Best backbone on validation: {best['backbone']}")
    print(f"    Best validation F1: {best['val_f1_macro']:.4f}")
    print(f"    Checkpoint: {best['checkpoint']}")

    # Variabili globali per le celle successive
    model  = best["model"]
    dm     = best["dm"]
    config = best["cfg"]

    # Grafico comparativo
    sorted_results = sorted(results, key=lambda x: x["val_f1_macro"])
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(
        [r["backbone"] for r in sorted_results],
        [r["val_f1_macro"] for r in sorted_results],
        color=["#4CAF50" if r["backbone"] == best["backbone"] else "#90CAF9" for r in sorted_results]
    )
    for bar, r in zip(bars, sorted_results):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f"{r['val_f1_macro']:.4f}", va="center", fontweight="bold")
    ax.set_xlim(0, 1.05)
    ax.set_xlabel("F1 macro (validation set)")
    ax.set_title("Ablation Study — Confronto backbone su validation", fontweight="bold")
    ax.axvline(0.9, color="red", linestyle="--", alpha=0.4, label="Soglia 0.9")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Nessun risultato. Esegui prima la cella di Ablation Study.")

## 10. Threshold Tuning sul validation set

La soglia decisionale di default (0.5) non è necessariamente ottimale per F1 macro con classi sbilanciate. Si ottimizza la threshold **solo sul validation set** (non sul test — evitare data leakage) e poi si applica al test nella sezione successiva.

In [ ]:
if results:
    device = next(model.parameters()).device
    val_probs, val_labels_th = collect_probs_and_labels(model, dm.val_dataloader(), device)

    # F1 con threshold default 0.50
    _, val_f1_default_05 = compute_binary_metrics(val_labels_th, val_probs, threshold=0.5)

    # Sweep di threshold sul validation set
    thresholds = np.arange(0.25, 0.76, 0.01)
    f1_scores_th = [
        f1_score(val_labels_th, (val_probs >= t).astype(int), average="macro", zero_division=0)
        for t in thresholds
    ]

    best_threshold = thresholds[np.argmax(f1_scores_th)]
    best_f1_val_thresholded = max(f1_scores_th)

    print("=== Threshold Tuning (validation set) ===")
    print(f"F1 macro validation con threshold 0.50:      {val_f1_default_05:.4f}")
    print(f"F1 macro validation con threshold ottimizzata: {best_f1_val_thresholded:.4f}")
    print(f"Threshold ottimale scelta: {best_threshold:.2f}")

    # Visualizzazione curva
    plt.figure(figsize=(9, 4))
    plt.plot(thresholds, f1_scores_th, color="steelblue", linewidth=2)
    plt.axvline(best_threshold, color="green", linestyle="--", label=f"Best thresh={best_threshold:.2f}")
    plt.axvline(0.5, color="red", linestyle="--", label="Default thresh=0.50")
    plt.xlabel("Threshold")
    plt.ylabel("F1 macro (val)")
    plt.title("Threshold Tuning sul Validation Set", fontweight="bold")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    best_threshold = 0.5
    print("Threshold default: 0.5")

## 11. Valutazione finale sul test set

**Unico uso del test set:** backbone scelto su validation, threshold su validation. Valutazione blind.

- Best backbone selezionato su validation
- Test F1 con threshold 0.50 e ottimizzata
- Classification Report e Confusion Matrix

In [ ]:
if results:
    device = next(model.parameters()).device
    all_probs, all_labels = collect_probs_and_labels(model, dm.test_dataloader(), device)

    # F1 con threshold default 0.50 e con threshold ottimizzata da validation
    _, f1_default = compute_binary_metrics(all_labels, all_probs, threshold=0.5)
    all_preds_opt, f1_opt = compute_binary_metrics(all_labels, all_probs, threshold=best_threshold)

    class_names = dm.train_ds.dataset.classes

    print("=== Valutazione finale sul test set (unico uso) ===")
    print(f"Best backbone selezionato su validation: {best['backbone']}")
    print(f"Test F1 con threshold 0.50:             {f1_default:.4f}")
    print(f"Test F1 con threshold ottimizzata ({best_threshold:.2f}): {f1_opt:.4f}")
    print()
    print("=== Classification Report (threshold ottimizzato) ===")
    print(classification_report(all_labels, all_preds_opt, target_names=class_names))

    # Confusion Matrix heatmap
    cm = confusion_matrix(all_labels, all_preds_opt)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=ax)
    ax.set_xlabel("Predetto", fontsize=12)
    ax.set_ylabel("Reale", fontsize=12)
    ax.set_title(f"Confusion Matrix — {best['backbone']}\nF1 macro = {f1_opt:.4f}", fontweight="bold")
    plt.tight_layout()
    plt.show()

## Model Calibration Analysis

**Analisi puramente diagnostica** — usa le probabilità già calcolate sul test set (Sezione 11). Non riaddestra il modello, non modifica backbone o threshold. Non influenza la selezione.

**Definizione usata (confidence della predizione):**
- `pred = (p >= best_threshold)` — predizione con threshold ottimizzata su validation
- `confidence = max(p, 1-p)` — probabilità assegnata alla classe predetta
- `correct = (pred == label)` — correttezza della predizione

La correttezza della predizione è calcolata utilizzando la threshold ottimizzata (`best_threshold`) trovata sul validation set, così che l'analisi di calibrazione rifletta esattamente il comportamento del modello finale.

- **Reliability Diagram:** asse X = confidence media del bin, asse Y = accuracy del bin; diagonale y=x = calibrazione perfetta
- **ECE:** media pesata di |accuracy − confidence| per bin

In [ ]:
def compute_ece(probs, labels, n_bins=10, threshold=0.5):
    """ECE: media pesata di |accuracy - confidence| per bin. Usa confidence della predizione."""
    probs = np.asarray(probs).flatten()
    labels = np.asarray(labels).flatten().astype(int)
    preds = (probs >= threshold).astype(int)  # predizione con threshold ottimizzata
    conf = np.maximum(probs, 1 - probs)  # confidence della predizione = max(p, 1-p)

    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (conf > bin_edges[i]) & (conf <= bin_edges[i + 1])
        if mask.sum() == 0:
            continue
        acc_bin = (preds[mask] == labels[mask]).mean()
        conf_bin = conf[mask].mean()
        weight = mask.sum() / len(labels)
        ece += weight * np.abs(acc_bin - conf_bin)
    return ece

def reliability_diagram(probs, labels, n_bins=10, ax=None, threshold=0.5):
    """Reliability Diagram: X=confidence media bin, Y=accuracy bin. Diagonale = calibrato."""
    probs = np.asarray(probs).flatten()
    labels = np.asarray(labels).flatten().astype(int)
    preds = (probs >= threshold).astype(int)  # predizione con threshold ottimizzata
    conf = np.maximum(probs, 1 - probs)  # confidence della predizione

    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs, bin_counts = [], [], []
    for i in range(n_bins):
        mask = (conf > bin_edges[i]) & (conf <= bin_edges[i + 1])
        if mask.sum() == 0:
            bin_accs.append(np.nan)
            bin_confs.append(np.nan)
            bin_counts.append(0)
        else:
            bin_accs.append((preds[mask] == labels[mask]).mean())
            bin_confs.append(conf[mask].mean())
            bin_counts.append(mask.sum())

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    bin_accs = np.array(bin_accs)
    bin_confs = np.array(bin_confs)
    valid = ~np.isnan(bin_accs)
    width = 0.04  # larghezza fissa per evitare sovrapposizioni
    ax.bar(bin_confs[valid], bin_accs[valid], width=width, alpha=0.7, color="steelblue", edgecolor="white", label="Accuracy (bin)")
    ax.plot([0, 1], [0, 1], "k--", linewidth=2, label="Perfettamente calibrato")
    ax.set_xlabel("Confidence media (bin)", fontsize=12)
    ax.set_ylabel("Accuracy (bin)", fontsize=12)
    ax.set_title("Reliability Diagram — Calibrazione del modello", fontsize=13, fontweight="bold")
    ax.legend(loc="lower right")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    return ax

# Usa probabilità già calcolate in Sezione 11 — nessun riaddestramento o modifica
if results and "all_probs" in globals() and "all_labels" in globals() and "best_threshold" in globals():
    ece = compute_ece(all_probs, all_labels, n_bins=10, threshold=best_threshold)
    fig, ax = plt.subplots(figsize=(6, 5))
    reliability_diagram(all_probs, all_labels, n_bins=10, ax=ax, threshold=best_threshold)
    plt.tight_layout()
    plt.show()

    print("=== Model Calibration ===")
    print(f"Expected Calibration Error (ECE): {ece:.4f}")
    print(f"(ECE basso = modello ben calibrato; ECE alto = confidence non riflette accuracy)")
else:
    print("Eseguire prima le Sezioni 10 e 11 per ottenere best_threshold, all_probs e all_labels.")


**Interpretazione:**

- Il **Reliability Diagram** ha asse X = confidence media del bin, asse Y = accuracy del bin. Un modello perfettamente calibrato ha le barre sulla diagonale y=x (tratteggiata).
- **ECE** misura la media pesata di |accuracy − confidence|. Valori bassi (<0.05) = buona calibrazione; valori alti (>0.1) = le probabilità non riflettono la confidenza reale.
- Si usa la **confidence della predizione** (max(p, 1-p)), con predizione basata su `best_threshold`.
- Questa analisi è **solo diagnostica**: non modifica il modello né la selezione.

## 12. TTA — Test Time Augmentation

**Analisi finale opzionale — NON influenza la selezione del modello.**

- Il **backbone** è già stato selezionato su validation (Sezione 8).
- La **threshold** è già stata ottimizzata su validation (Sezione 10).
- La TTA è una **valutazione opzionale** sul test del modello finale già definito.
- **Non viene utilizzata** per selezionare configurazioni, backbone o iperparametri.

Confronto: test F1 senza TTA vs test F1 con TTA (4 flip, media probabilità). Usa `best_threshold`.

In [ ]:
def predict_tta(model, loader, device, threshold=0.5):
    """Test Time Augmentation: media su 4 flip. Solo per metriche aggiuntive, non modifica selezione."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            p0 = torch.sigmoid(model(x))
            p1 = torch.sigmoid(model(torch.flip(x, [-1])))    # flip orizzontale
            p2 = torch.sigmoid(model(torch.flip(x, [-2])))    # flip verticale
            p3 = torch.sigmoid(model(torch.flip(x, [-2,-1]))) # entrambi
            p  = (p0 + p1 + p2 + p3) / 4
            all_probs.extend(p.cpu().numpy().tolist())
            all_labels.extend(y.tolist())
    preds = (np.array(all_probs) >= threshold).astype(int)
    return preds, np.array(all_labels)

# TTA: solo metriche aggiuntive. Modello e best_threshold già fissati su validation.
# NON aggiorna best, best_threshold, results — nessuna influenza sulla selezione.
if results:
    device = next(model.parameters()).device
    preds_tta, labels_tta = predict_tta(model, dm.test_dataloader(), device, threshold=best_threshold)
    f1_tta = f1_score(labels_tta, preds_tta, average="macro", zero_division=0)

    print("=== Confronto TTA (solo metriche, nessuna selezione) ===")
    if "f1_opt" in globals():
        print(f"Test F1 senza TTA (thresh ottimizzato): {f1_opt:.4f}")
    else:
        print("(Eseguire Sezione 11 per il confronto senza TTA)")
    print(f"Test F1 con TTA    (thresh ottimizzato): {f1_tta:.4f}")
    if "f1_opt" in globals():
        gain = f1_tta - f1_opt
        print(f"Guadagno TTA: {gain:+.4f}")

## 13. Grad-CAM — Visualizzazione interpretabilità

GradCAM mostra su quali regioni dell'immagine si focalizza il modello per la classificazione. Permette di verificare che il modello stia guardando i fiori e non artefatti di background.

In [ ]:
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    GRADCAM_AVAILABLE = True
except ImportError:
    GRADCAM_AVAILABLE = False
    print("grad-cam non installato. Installa con: pip install grad-cam")


class BinaryClassifierTarget:
    """Target custom per modelli con output scalare (classificazione binaria)."""
    def __init__(self, category):
        self.category = category  # 0 o 1

    def __call__(self, model_output):
        out = model_output.flatten()
        if len(out) == 1:
            return out[0] if self.category == 1 else -out[0]
        return out[self.category]


if results and GRADCAM_AVAILABLE:

    def get_target_layer(model, backbone_name):
        if "resnet" in backbone_name:
            return [model.backbone.layer4[-1]]
        elif "efficientnet" in backbone_name:
            return [model.backbone.conv_head]
        elif "convnext" in backbone_name:
            return [model.backbone.stages[-1].blocks[-1]]
        return [list(model.backbone.modules())[-3]]

    device = next(model.parameters()).device
    target_layers = get_target_layer(model, best["backbone"])
    cam = GradCAM(model=model, target_layers=target_layers)

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    ds = dm.val_ds  # Usa validation, non test — Grad-CAM non deve "peekare" nel test
    mean_arr = np.array(dm.mean)
    std_arr  = np.array(dm.std)

    model.eval()
    shown = {0: 0, 1: 0}

    for idx in range(len(ds)):
        img_tensor, label = ds[idx]

        if shown[label] >= 3:
            if shown[0] >= 3 and shown[1] >= 3:
                break
            continue

        input_tensor = img_tensor.unsqueeze(0).to(device)

        # Predizione
        with torch.no_grad():
            output = model(input_tensor)
            pred_class = int((torch.sigmoid(output.flatten()[0]) > 0.5).long().item())

        # Grad-CAM con target custom binario
        grayscale_cam = cam(
            input_tensor=input_tensor,
            targets=[BinaryClassifierTarget(pred_class)]
        )[0]

        # Denormalizza per visualizzazione
        img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
        img_np = (img_np * std_arr + mean_arr).clip(0, 1).astype(np.float32)
        visualization = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

        row = label
        col = shown[label]
        axes[row][col].imshow(visualization)

        correct = "✓" if pred_class == label else "✗"
        axes[row][col].set_title(
            f"GT: {classes[label]}  {correct}\nPred: {classes[pred_class]}",
            fontsize=9,
            color="green" if pred_class == label else "red"
        )
        axes[row][col].axis("off")
        shown[label] += 1

    # Etichette righe
    for row, cls_name in enumerate(classes):
        axes[row][0].set_ylabel(cls_name, fontsize=11, fontweight="bold", rotation=90)

    plt.suptitle(
        f"Grad-CAM — {best['backbone']}\nZone di attenzione del modello",
        fontsize=13, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()

## 14. Conclusioni

Riepilogo finale: Best backbone on validation, Best validation F1, Best threshold, Test F1 (default e ottimizzata), Test F1 with TTA.

In [ ]:
print("=" * 65)
print("  FLOWER RECOGNITION — GreenTech Solutions Ltd.")
print("  Progetto completato")
print("=" * 65)
print(f"  Backbones testati:              {len(cfg_global.BACKBONES)}")
print(f"  Checkpoint dir:                 {cfg_global.CHECKPOINT_DIR}")

if results:
    print(f"  Best backbone on validation:   {best['backbone']}")
    print(f"  Best validation F1:             {best['val_f1_macro']:.4f}")
    print(f"  Best threshold from validation: {best_threshold:.2f}")
    print(f"  Test F1 (threshold 0.50):       {f1_default:.4f}")
    print(f"  Test F1 (threshold ottimizzata): {f1_opt:.4f}")
    try:
        print(f"  Test F1 with TTA:                 {f1_tta:.4f}")
    except NameError:
        pass
    try:
        if GRADCAM_AVAILABLE:
            print("  Grad-CAM:             eseguito")
    except NameError:
        pass
    print()
    print("  Per resume dopo crash:")
    print("    cfg_global.RESUME_BACKBONE = '<backbone_name>'")
    print("    cfg_global.RESUME_CKPT     = 'checkpoints/<backbone>/periodic/last.ckpt'")
    print("    cfg_global.BACKBONES       = ['<backbone_name>']")
print("=" * 65)

## 15. Documentazione del progetto

Il notebook costituisce l’artefatto eseguibile principale del progetto e contiene l’intera pipeline di sviluppo:

- esplorazione del dataset (EDA)
- preprocessing e data augmentation
- definizione del modello e strategia di training
- ablation study sui backbone
- threshold tuning
- valutazione finale sul test set
- interpretabilità tramite Grad-CAM

Per una descrizione completa dell’architettura, delle scelte metodologiche e dei risultati sperimentali sono disponibili i seguenti documenti:

- **README tecnico del progetto**  
  https://drive.google.com/file/d/10a6lEgaBEZDLJQW_rTW_W7s5k_GCp-jd/view?usp=sharing

- **Relazione tecnica completa (PDF)**  
  https://drive.google.com/file/d/1WiM9TTK4hnwrjl4nRSVWD42gLs16ST6R/view?usp=sharing

La documentazione include:
- descrizione del dataset e delle sue limitazioni
- dettagli della pipeline di training
- confronto tra architetture CNN (ResNet, EfficientNet, ConvNeXt)
- analisi delle metriche di valutazione
- discussione su robustezza, generalizzazione e possibili sviluppi futuri

---

**Flower Recognition — GreenTech Solutions Ltd.**  
Progetto sviluppato nell’ambito del *Master in AI Engineering — Computer Vision Module*